# ESM Embedding Generation (Per-Token)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import scipy.stats
from torch.utils.data import DataLoader
from tqdm import tqdm
import os, glob, ast, joblib, gc,esm, torch
from joblib import Parallel, delayed

In [2]:
# ------------------------------------------------------------
# Embedding Generation Configuration (ESM-2)
# Model: facebook/esm2_t33_650M_UR50D
# CPU: 12 cores (used for parallel preprocessing with joblib)
# GPU: 2 × A100 40GB (used for ESM model inference)
# Tokenization: ESM tokenizer with BOS/EOS and padding
# Output Format: 
#   - mean/ : per-sequence mean embeddings, shape (N, 320)
#   - token/: full token-level embeddings, shape (N, L_PAD, 320)
# ------------------------------------------------------------


In [3]:
# ------------------------------------------------------------
#  generate_embeddings(): save BOTH mean and per‑token tensors
#                *token file now fixed‑shape, no object array*
# ------------------------------------------------------------

MODEL_PATH = "/datasets/bio/esm/models/esm2_t6_8M_UR50D.pt"
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
EMBED_DIM  = 320

def generate_embeddings(
        df_chunk: pd.DataFrame,
        ref_protein: str,
        chunk_id: int,
        gene_name: str,
        truncate_len: int,
):
    """Write two compressed .npz files for this chunk and return nothing."""
    # ---------- load model once per kernel ------------------------
    global _ESM_MODEL, _BATCH_CONV
    if "_ESM_MODEL" not in globals():
        _ESM_MODEL, alphabet = esm.pretrained.load_model_and_alphabet_local(MODEL_PATH)
        _ESM_MODEL.eval().to(DEVICE)
        _BATCH_CONV = alphabet.get_batch_converter()

    L_PAD = truncate_len            # fixed padded length for this gene

    mean_vecs, tok_tensors, ids = [], [], []

    data   = list(zip(df_chunk["Filename"], df_chunk["Protein_Sequence"]))
    loader = DataLoader(data, batch_size=32, shuffle=False)

    with torch.no_grad():
        for labels_b, seqs_b in tqdm(loader, leave=False):
            lbls, strs, toks = _BATCH_CONV(list(zip(labels_b, seqs_b)))
            toks = toks.to(DEVICE)

            reps = _ESM_MODEL(toks, repr_layers=[_ESM_MODEL.num_layers])[
                       "representations"][_ESM_MODEL.num_layers].cpu()

            for i, seq in enumerate(strs):
                L = min(len(seq), L_PAD)
                tok_repr = reps[i, 1:L+1]            # (L, 320)

                # ---------- mean (skip X) ---------------------------
                valid = [aa != "X" for aa in seq[:L]]
                mean_vecs.append(tok_repr[valid].mean(0).numpy())

                # ---------- per‑token, pad/truncate to L_PAD -------
                if not all(valid):
                    tok_repr[~torch.tensor(valid)] = 0.0

                if tok_repr.shape[0] < L_PAD:        # pad shorter seq
                    pad = torch.zeros(L_PAD, EMBED_DIM)
                    pad[:tok_repr.shape[0]] = tok_repr
                    tok_tensors.append(pad.numpy())
                else:                                # already L_PAD
                    tok_tensors.append(tok_repr.numpy())

            ids.extend(lbls)
    # Create directories if they don't exist
    os.makedirs(f"embeddings/{gene_name}_ETH/mean", exist_ok=True)
    os.makedirs(f"embeddings/{gene_name}_ETH/token", exist_ok=True)

    # ---------- write files ---------------------------------------
    np.savez_compressed(
        f"data/embeddings/{gene_name}/mean/{gene_name}_mean_chunk_{chunk_id}.npz",
        identifier=np.array(ids),
        mean_embedding=np.vstack(mean_vecs).astype("float32")
    )

    np.savez_compressed(
        f"data/embeddings/{gene_name}/token/{gene_name}_tok_chunk_{chunk_id}.npz",
        identifier=np.array(ids),
        tokens=np.stack(tok_tensors).astype("float32")  # (n, L_PAD, 320)
    )

    # ---------- clean up ------------------------------------------
    del reps, tok_repr, mean_vecs, tok_tensors
    torch.cuda.empty_cache()
    gc.collect()


## select protein-coding gene

In [4]:
gene_name='tlyA'
# pick the reference CDS protein sequence
ref_seqs=pd.read_csv('./data/catalog/protein_sequences.csv')
ref_protein = ref_seqs.loc[ref_seqs['gene'] == gene_name, 'protein_sequence'].values

In [5]:
#load the gene sequence file
gene_sequences=pd.read_csv(f"./data/sequence_data_csv/{gene_name}_combined_sequence_data.csv")
gene_sequences_filtered = gene_sequences[gene_sequences["Frameshift_Mutation"] == 0].copy()
# Ensure required columns exist
if "Protein_Sequence" not in gene_sequences_filtered.columns:
    raise ValueError("CSV file must contain a 'Protein_Sequence' column.")

In [6]:
# sanity check: check for invalid characters. if X >0.05 per sequence, drop that sequence
valid_amino_acids = set("ACDEFGHIKLMNPQRSTVWY")  # Standard 20 amino acids

# Check each sequence
for idx, seq in enumerate(gene_sequences_filtered["Protein_Sequence"]):
    invalid_chars = [c for c in seq if c not in valid_amino_acids]
    if invalid_chars:
        print(f"WARNING: Sequence {idx} contains invalid characters: {invalid_chars}")
        
# filter out highly ambiguous sequences
max_x_fraction = 0.05
valid_indices = []

for i, seq in enumerate(gene_sequences_filtered["Protein_Sequence"]):
    num_x = seq.count("X")
    if num_x / len(seq) <= max_x_fraction:
        valid_indices.append(i)
    else:
        print(f"Dropping sequence {i}: {num_x}/{len(seq)} Xs")

gene_sequences_filtered = gene_sequences_filtered.iloc[valid_indices].reset_index(drop=True)



In [7]:
len(gene_sequences_filtered)
gene_sequences_filtered["Filename"].value_counts().head()


Filename
SAMEA3558278    1
SAMEA3562771    1
SAMN03648611    1
SAMN03647522    1
SAMN08892448    1
Name: count, dtype: int64

## generate embeddings and save memory-efficient chunks

In [8]:
chunk_size = 1000
for k, start in enumerate(range(0, len(gene_sequences_filtered), chunk_size)):
    df_chunk = gene_sequences_filtered.iloc[start:start+chunk_size]
    generate_embeddings(df_chunk, ref_protein,
                        chunk_id=k, gene_name=gene_name, truncate_len=len(ref_protein[0]))
